# GPU 加速運算概覽與環境建置

> 本 Notebook 是 GPU 加速系列的起點，帶你了解 RAPIDS 生態系全貌，並建置好執行環境。

## GPU 加速系列 Notebooks

1. [Numba CUDA 基礎](./01-Introduction-to-cuda-python-with-numba.ipynb)
2. **[GPU 加速概覽與環境建置](./02-GPU-Acceleration-Overview-and-Environment-Setup.ipynb) ← 目前位置**
3. [CuPy - GPU 版 NumPy](./03-CuPy-GPU-Accelerated-NumPy.ipynb)
4. [cuDF - GPU 版 pandas](./04-cuDF-GPU-Accelerated-DataFrames.ipynb)
5. [cuML - GPU 版 scikit-learn](./05-cuML-GPU-Accelerated-Machine-Learning.ipynb)
6. [cuGraph - GPU 版 NetworkX](./06-cuGraph-GPU-Accelerated-Graph-Analytics.ipynb)
7. [Dask-CUDA 多 GPU 處理](./07-Dask-CUDA-Multi-GPU-and-Large-Scale-Processing.ipynb)
8. [RAPIDS 端到端實戰](./08-RAPIDS-End-to-End-Data-Analysis-Pipeline.ipynb)

---
## 1. 為什麼需要 GPU 加速？

### CPU vs GPU 架構差異

| 特性 | CPU | GPU |
|------|-----|-----|
| 核心數 | 4-64 個強大核心 | 數千個輕量核心 |
| 擅長 | 複雜邏輯、分支判斷、序列任務 | 大量相同運算的平行處理 |
| 記憶體 | 系統記憶體 (RAM, 通常 16-128 GB) | 顯示記憶體 (VRAM, 通常 8-80 GB) |
| 類比 | 一位數學教授 | 一千位小學生同時算數學 |

**關鍵觀念：** 資料分析中的許多操作（向量運算、矩陣乘法、DataFrame groupby）本質上是「對大量資料做相同運算」，非常適合 GPU 平行處理。

```
CPU 處理 1000 萬筆資料：
┌─────┐
│ Core│ → 逐一處理 → ████████████████████████ (慢但靈活)
└─────┘

GPU 處理 1000 萬筆資料：
┌──┐┌──┐┌──┐┌──┐
│C1││C2││C3││..│ → 同時處理 → ████ (快但需要平行化)
└──┘└──┘└──┘└──┘
  × 數千個核心
```

---
## 2. RAPIDS 生態系全景圖

[RAPIDS](https://rapids.ai/) 是 NVIDIA 維護的開源 GPU 加速資料科學套件。它的核心設計理念是：**讓你用熟悉的 API，獲得 GPU 加速**。

### CPU → GPU 對應關係

| 你熟悉的 CPU 套件 | RAPIDS GPU 版本 | 加速倍數 | 用途 |
|-------------------|-----------------|----------|------|
| **NumPy** | **CuPy** | 100x+ | 陣列 / 矩陣運算 |
| **pandas** | **cuDF** | 30-150x | DataFrame 資料處理 |
| **scikit-learn** | **cuML** | 5-175x | 機器學習演算法 |
| **NetworkX** | **cuGraph** | ~39x | 圖形 / 網路分析 |
| **SciPy (signal)** | **CuPy** | 50x+ | 訊號處理 |
| **Dask** | **Dask-CUDA** | 線性擴展 | 多 GPU / 分散式運算 |

### 生態系架構

```
┌─────────────────────────────────────────────────┐
│                  你的 Python 程式碼               │
├────────┬────────┬─────────┬──────────┬──────────┤
│  cuDF  │  cuML  │ cuGraph │   CuPy   │ Dask-CUDA│
│(pandas)│(sklearn)│(NetworkX)│ (NumPy) │ (分散式) │
├────────┴────────┴─────────┴──────────┴──────────┤
│                    CUDA Toolkit                   │
├─────────────────────────────────────────────────┤
│                  NVIDIA GPU 硬體                  │
└─────────────────────────────────────────────────┘
```

### Zero-Code-Change 加速（RAPIDS 25.02+）

最新版 RAPIDS 支援 **零程式碼修改** 加速：
- `%load_ext cudf.pandas` → 你的 pandas 程式碼自動在 GPU 上執行
- `cuml.accel.install()` → scikit-learn 程式碼自動用 GPU
- `nx-cugraph` backend → NetworkX 自動用 GPU

也就是說，**你可能不需要改任何一行程式碼就能獲得加速！**

---
## 3. 硬體需求

### GPU 最低需求

| 需求 | 說明 |
|------|------|
| GPU 品牌 | **NVIDIA** (AMD / Intel 不支援 RAPIDS) |
| Compute Capability | **7.0 以上** (Volta 架構或更新) |
| VRAM | 建議 **16 GB 以上** |
| CUDA Toolkit | 需與 RAPIDS 版本對應 |

### 常見 GPU 對照

| GPU | Compute Capability | VRAM | 適用場景 |
|-----|-------------------|------|----------|
| Tesla T4 | 7.5 | 16 GB | 入門 / 雲端免費 (Colab) |
| RTX 3090 | 8.6 | 24 GB | 個人工作站 |
| A100 | 8.0 | 40/80 GB | 專業 / 生產環境 |
| H100 | 9.0 | 80 GB | 頂級運算 |

> **沒有 NVIDIA GPU？別擔心！** 下一節介紹免費取得 GPU 的方式。

---
## 4. 如何取得 GPU 環境

### 方式一：Google Colab（推薦入門）

1. 前往 [colab.research.google.com](https://colab.research.google.com)
2. 開啟 Notebook → 選單 `執行階段` → `變更執行階段類型` → 選擇 **T4 GPU**
3. RAPIDS cuDF 已預裝（自 2025 年起），其他套件需 `!pip install`

### 方式二：Kaggle Notebooks

1. 前往 [kaggle.com/code](https://www.kaggle.com/code)
2. 新增 Notebook → 右側 Settings → Accelerator → **GPU T4 x2**
3. 每週約 30 小時免費 GPU 額度

### 方式三：雲端服務

| 平台 | GPU 選項 | 適合 |
|------|----------|------|
| GCP (Vertex AI) | T4, A100, H100 | 專案開發 |
| AWS (SageMaker) | T4, A10G, A100 | 企業環境 |
| Azure ML | T4, A100 | 微軟生態系 |

### 方式四：本地 NVIDIA GPU

如果你的電腦有 NVIDIA GPU（RTX 系列）：
1. 安裝最新 [NVIDIA 驅動程式](https://www.nvidia.com/Download/index.aspx)
2. 使用 conda 安裝 RAPIDS（見下節）

---
## 5. 環境建置與安裝

### 5.1 檢查 GPU 狀態

執行以下 cell 確認你的環境是否有可用的 NVIDIA GPU：

In [ ]:
# 環境檢查 - GPU 可用性
import subprocess
import shutil

if shutil.which('nvidia-smi'):
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print(result.stdout)
    GPU_AVAILABLE = True
else:
    print('='*60)
    print('未偵測到 NVIDIA GPU')
    print('='*60)
    print()
    print('建議使用以下免費 GPU 環境：')
    print('  1. Google Colab: https://colab.research.google.com')
    print('  2. Kaggle:       https://www.kaggle.com/code')
    print()
    print('本 Notebook 的說明內容仍可閱讀學習。')
    GPU_AVAILABLE = False

### 5.2 安裝 RAPIDS

根據你的環境選擇安裝方式：

In [ ]:
# ===== 方式一：conda 安裝（推薦）=====
# 在終端機執行：
# conda install -c rapidsai -c conda-forge cudf cuml cugraph cupy

# ===== 方式二：pip 安裝（CUDA 12）=====
# !pip install cudf-cu12 cuml-cu12 cugraph-cu12 cupy-cuda12x

# ===== 方式三：Google Colab =====
# cuDF 已預裝，其他套件：
# !pip install cuml-cu12 cugraph-cu12

# ===== 方式四：Docker =====
# docker pull rapidsai/rapidsai:latest
# docker run --gpus all -p 8888:8888 rapidsai/rapidsai:latest

print('請根據你的環境取消註解對應的安裝指令')

### 5.3 驗證安裝

In [ ]:
# 驗證 RAPIDS 套件是否安裝成功
packages = {
    'cupy': 'CuPy (GPU NumPy)',
    'cudf': 'cuDF (GPU pandas)',
    'cuml': 'cuML (GPU scikit-learn)',
    'cugraph': 'cuGraph (GPU NetworkX)',
}

print('RAPIDS 套件安裝狀態：')
print('-' * 45)

for pkg, desc in packages.items():
    try:
        mod = __import__(pkg)
        ver = getattr(mod, '__version__', '已安裝')
        print(f'  {desc:30s} v{ver}')
    except ImportError:
        print(f'  {desc:30s} 未安裝')

---
## 6. 快速體驗：NumPy vs CuPy 效能比較

讓我們用一個簡單的矩陣運算來體驗 GPU 加速的威力：

In [ ]:
import numpy as np
import time

# 矩陣大小
N = 5000

# ===== CPU 版本 (NumPy) =====
a_cpu = np.random.rand(N, N).astype(np.float32)
b_cpu = np.random.rand(N, N).astype(np.float32)

start = time.time()
c_cpu = np.dot(a_cpu, b_cpu)        # CPU 矩陣乘法
cpu_time = time.time() - start

print(f'CPU (NumPy) 矩陣乘法 ({N}x{N}): {cpu_time:.4f} 秒')

In [ ]:
# ===== GPU 版本 (CuPy) =====
if GPU_AVAILABLE:
    try:
        import cupy as cp

        a_gpu = cp.random.rand(N, N, dtype=cp.float32)
        b_gpu = cp.random.rand(N, N, dtype=cp.float32)

        # 暖機（第一次執行會編譯 kernel，不計入時間）
        _ = cp.dot(a_gpu, b_gpu)
        cp.cuda.Stream.null.synchronize()  # 等待 GPU 完成

        # 正式計時
        start = time.time()
        c_gpu = cp.dot(a_gpu, b_gpu)       # GPU 矩陣乘法
        cp.cuda.Stream.null.synchronize()  # 等待 GPU 完成
        gpu_time = time.time() - start

        print(f'GPU (CuPy)  矩陣乘法 ({N}x{N}): {gpu_time:.4f} 秒')
        print(f'加速倍數: {cpu_time / gpu_time:.1f}x')

        # 驗證結果正確性
        diff = float(cp.max(cp.abs(c_gpu - cp.asarray(c_cpu))))
        print(f'最大誤差: {diff:.6f} (浮點精度範圍內)')

    except ImportError:
        print('CuPy 未安裝。請先執行安裝步驟。')
else:
    print('需要 GPU 環境才能執行此 cell。')
    print('預期結果：GPU 版本約快 50-100 倍')

### 效能差異解讀

矩陣乘法是 GPU 最擅長的運算之一：
- 每個輸出元素可以獨立計算（高度平行化）
- 運算密度高（大量浮點乘法與加法）
- 資料重用率高（GPU 快取效益大）

> **注意：** 不是所有運算都能獲得這樣的加速。GPU 適合「大量資料 × 簡單運算」的場景。
> 小資料量或複雜分支邏輯的任務，CPU 可能更快。

---
## 7. GPU 加速的適用場景

### 適合 GPU 加速

- 資料量大（> 10 萬筆以上的 DataFrame 操作）
- 矩陣 / 向量運算（線性代數、統計計算）
- 機器學習訓練（尤其是大資料集 + 多次迭代）
- 圖形分析（大型社群網路、推薦系統）
- 批次資料處理（ETL pipeline）

### 不適合 GPU 加速

- 小資料量（< 1 萬筆，CPU-GPU 傳輸成本 > 運算加速）
- 大量分支判斷的邏輯（if/else 密集的演算法）
- 需要頻繁 CPU-GPU 資料傳輸的工作流
- 字串處理為主的任務（GPU 字串支援有限）
- 記憶體超過 GPU VRAM 的資料集（需搭配 Dask-CUDA）

### 決策流程

```
資料量 > 10 萬筆？
  ├─ 否 → 用 pandas / NumPy (CPU 就夠快)
  └─ 是 → 運算是否可平行化？
            ├─ 否 → 用 CPU
            └─ 是 → 資料量 < GPU VRAM？
                      ├─ 是 → 用 cuDF / cuML / CuPy (單 GPU)
                      └─ 否 → 用 Dask-CUDA (多 GPU / 分散式)
```

---
## 8. 重要觀念：CPU-GPU 資料傳輸

使用 GPU 加速時，最容易忽略的效能瓶頸是 **資料在 CPU 記憶體和 GPU 記憶體之間的傳輸時間**。

```
┌──────────┐    PCIe 匯流排    ┌──────────┐
│  CPU RAM  │ ←──────────────→ │ GPU VRAM │
│ (Host)    │   ~12-32 GB/s    │ (Device) │
└──────────┘                   └──────────┘
```

### 最佳實踐

1. **盡量減少傳輸次數**：一次傳入大量資料，在 GPU 上完成所有運算後再傳回
2. **避免頻繁來回**：不要每個小步驟都在 CPU 和 GPU 之間切換
3. **使用 RAPIDS 全套件**：cuDF → cuML → cuGraph 全程在 GPU 上，避免中間傳輸
4. **最後才轉回 CPU**：只在需要視覺化（matplotlib）或輸出時才傳回

```python
# 不好的做法 (頻繁傳輸)
df_gpu = cudf.from_pandas(df)     # CPU → GPU
result = df_gpu.groupby('A').sum()
result_cpu = result.to_pandas()    # GPU → CPU
result_cpu['new'] = ...            # 在 CPU 處理
df_gpu2 = cudf.from_pandas(result_cpu)  # CPU → GPU (又傳一次！)

# 好的做法 (全程 GPU)
df_gpu = cudf.from_pandas(df)     # CPU → GPU（只傳一次）
result = df_gpu.groupby('A').sum()
result['new'] = ...                # 在 GPU 上處理
final = result.to_pandas()         # GPU → CPU（最後才傳回）
```

---
## 9. 本系列學習路線建議

| 你的背景 | 建議學習順序 |
|----------|-------------|
| **資料分析師** (pandas 為主) | 02 → 04 (cuDF) → 05 (cuML) → 08 (實戰) |
| **機器學習工程師** | 02 → 03 (CuPy) → 05 (cuML) → 07 (Dask) → 08 (實戰) |
| **科學計算 / 數值分析** | 02 → 03 (CuPy) → 01 (Numba CUDA) |
| **圖形 / 網路分析** | 02 → 06 (cuGraph) |
| **全面學習** | 02 → 03 → 04 → 05 → 06 → 07 → 08 |

### 參考資源

- [RAPIDS 官方文件](https://docs.rapids.ai/)
- [RAPIDS GitHub](https://github.com/rapidsai)
- [NVIDIA RAPIDS 安裝指南](https://docs.rapids.ai/install/)
- [RAPIDS Release Notes](https://docs.rapids.ai/releases/schedule/)
- [CuPy 官方文件](https://cupy.dev/)
- [NVIDIA Deep Learning Institute 免費課程](https://www.nvidia.com/en-us/training/)

---

> **下一步：** 前往 [03-CuPy-GPU-Accelerated-NumPy.ipynb](./03-CuPy-GPU-Accelerated-NumPy.ipynb) 開始學習 CuPy，或直接跳到 [04-cuDF-GPU-Accelerated-DataFrames.ipynb](./04-cuDF-GPU-Accelerated-DataFrames.ipynb) 學習 GPU 版 pandas。